# Kaggle 06. Decoder-only LLM Prompting Baseline

Этот ноутбук проверяет decoder-only модели без обучения: zero-shot, few-shot и reasoning prompt.
По умолчанию запускается только `zero_shot`. Для следующих прогонов добавьте режимы
`few_shot` и `reasoning` в `PROMPT_MODES`.

Вход: обычный русский текст `text_ru`, не маскированный `text_masked`.
Выход: одна из 9 канонических меток.

In [ ]:
%pip install -q -U "transformers>=5.5.4" accelerate bitsandbytes sentencepiece scikit-learn seaborn kagglehub

In [ ]:
import json
import logging
import random
import re
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('kaggle_decoder_only_prompting')
sns.set_theme(style='whitegrid')

In [ ]:
INPUT_JSONL_CANDIDATES = sorted(Path('/kaggle/input').rglob('*.jsonl'))
assert INPUT_JSONL_CANDIDATES, 'Upload cocolofa_ru_v2.jsonl, cocolofa_ru_v2_masked.jsonl, or cocolofa_ru_v2_explanations.jsonl to /kaggle/input first.'

for path in INPUT_JSONL_CANDIDATES:
    print(path)

EXPLANATIONS_DATASET_PATH = next(
    (path for path in INPUT_JSONL_CANDIDATES if path.name == 'cocolofa_ru_v2_explanations.jsonl'),
    None,
)
RAW_DATASET_PATH = next(
    (path for path in INPUT_JSONL_CANDIDATES if path.name == 'cocolofa_ru_v2.jsonl'),
    None,
)
MASKED_DATASET_PATH = next(
    (path for path in INPUT_JSONL_CANDIDATES if path.name == 'cocolofa_ru_v2_masked.jsonl'),
    None,
)
DATASET_PATH = EXPLANATIONS_DATASET_PATH or RAW_DATASET_PATH or MASKED_DATASET_PATH
assert DATASET_PATH is not None, 'No supported COCOLOFA-RU dataset was found.'

OUTPUT_ROOT = Path('/kaggle/working/phase3_decoder_only_prompting')
TEXT_COLUMN = 'text_ru'
SEED = 42

# Kaggle model page:
# https://www.kaggle.com/models/google/gemma-4/transformers/gemma-4-26b-a4b
KAGGLE_MODEL_HANDLE = 'google/gemma-4/transformers/gemma-4-26b-a4b'
KAGGLE_MODEL_LOCAL_PATH = None
HF_FALLBACK_MODEL = 'google/gemma-4-26B-A4B-it'

# First run: zero-shot only. Later change to ['zero_shot', 'few_shot', 'reasoning'].
PROMPT_MODES = ['zero_shot']
EVAL_SPLIT = 'test'
MAX_EVAL_ROWS = 100
EVAL_BATCH_SIZE = 1
MAX_INPUT_LENGTH = 1536
MAX_NEW_TOKENS = 96
FEW_SHOT_PER_CLASS = 1

{
    'dataset_path': str(DATASET_PATH),
    'text_column': TEXT_COLUMN,
    'kaggle_model_handle': KAGGLE_MODEL_HANDLE,
    'prompt_modes': PROMPT_MODES,
    'eval_split': EVAL_SPLIT,
    'max_eval_rows': MAX_EVAL_ROWS,
}

In [ ]:
LABELS = [
    'none',
    'appeal to authority',
    'appeal to majority',
    'appeal to nature',
    'appeal to tradition',
    'appeal to worse problems',
    'false dilemma',
    'hasty generalization',
    'slippery slope',
]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
N_CLASSES = len(LABELS)
ACCEPTED_TRANSLATION_STATUSES = {'ok', 'repaired_ok'}

LABEL_DESCRIPTIONS_RU = {
    'none': 'логической ошибки из списка нет',
    'appeal to authority': 'вывод принимается из-за авторитета источника вместо аргументов',
    'appeal to majority': 'вывод принимается из-за популярности мнения',
    'appeal to nature': 'естественность или неестественность объявляется доказательством правильности',
    'appeal to tradition': 'вывод обосновывается тем, что так принято или так было всегда',
    'appeal to worse problems': 'проблема обесценивается ссылкой на более серьёзные проблемы',
    'false dilemma': 'ситуация искусственно сведена к двум вариантам',
    'hasty generalization': 'общий вывод делается по недостаточному числу случаев',
    'slippery slope': 'утверждается цепочка всё более тяжёлых последствий без достаточного обоснования',
}

SYSTEM_PROMPT = (
    'Ты строгий классификатор логических ошибок. '
    'Работай только с заданным списком меток. '
    'Не придумывай новые классы.'
)


def normalize_label(label: str) -> str:
    return ' '.join(str(label).strip().lower().replace('_', ' ').replace('-', ' ').split())


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False))
            handle.write('\n')


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def parse_json_records(path: Path) -> list[dict]:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        return []

    rows = []
    decoder = json.JSONDecoder()
    cursor = 0
    while cursor < len(text):
        while cursor < len(text) and text[cursor].isspace():
            cursor += 1
        if cursor >= len(text):
            break
        parsed, next_cursor = decoder.raw_decode(text, cursor)
        if isinstance(parsed, list):
            rows.extend(parsed)
        else:
            rows.append(parsed)
        cursor = next_cursor
        while cursor < len(text) and text[cursor] in ',\n\r\t ':
            cursor += 1
    return rows


def load_cocolofa_jsonl(path: Path) -> pd.DataFrame:
    rows = parse_json_records(path)
    df = pd.DataFrame(rows)
    assert 'label_str' in df.columns, 'label_str column is required.'
    assert 'sample_id' in df.columns, 'sample_id column is required.'
    assert 'split' in df.columns, 'split column is required.'
    assert TEXT_COLUMN in df.columns, f'{TEXT_COLUMN} column is required.'

    df = df.copy()
    df['label_str'] = df['label_str'].map(normalize_label)
    df['label_id'] = pd.to_numeric(df.get('label_id'), errors='coerce').fillna(df['label_str'].map(LABEL_TO_ID)).astype(int)
    df['sample_id'] = pd.to_numeric(df['sample_id'], errors='coerce').astype(int)
    df['split'] = df['split'].astype(str).str.lower().str.strip()
    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna('').astype(str).str.strip()
    if 'translation_status' in df.columns:
        df = df[df['translation_status'].isin(ACCEPTED_TRANSLATION_STATUSES)]
    df = df[df['label_str'].isin(LABEL_TO_ID)]
    df = df[df[TEXT_COLUMN].str.len() > 0]
    return df.reset_index(drop=True)

In [ ]:
def find_local_kaggle_model_dir() -> str | None:
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    keywords = {'gemma', '4', '26b', 'a4b'}
    for root in roots:
        if not root.exists():
            continue
        for config_path in root.rglob('config.json'):
            parts = {part.lower() for part in config_path.parts}
            joined = '/'.join(part.lower() for part in config_path.parts)
            if all(keyword in joined for keyword in keywords) or 'gemma-4-26b-a4b' in joined:
                return str(config_path.parent)
    return None


def resolve_model_path() -> str:
    explicit = Path(KAGGLE_MODEL_LOCAL_PATH) if KAGGLE_MODEL_LOCAL_PATH else None
    if explicit and explicit.exists():
        return str(explicit)

    local_model_dir = find_local_kaggle_model_dir()
    if local_model_dir:
        logger.info('Using local Kaggle model directory: %s', local_model_dir)
        return local_model_dir

    try:
        import kagglehub

        logger.info('Downloading Kaggle model via kagglehub: %s', KAGGLE_MODEL_HANDLE)
        return kagglehub.model_download(KAGGLE_MODEL_HANDLE)
    except Exception as exc:
        logger.warning('KaggleHub model download failed: %s', exc)
        logger.warning('Falling back to Hugging Face model id: %s', HF_FALLBACK_MODEL)
        return HF_FALLBACK_MODEL


def load_model_and_tokenizer(model_path: str):
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    tokenizer = getattr(processor, 'tokenizer', processor)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {'trust_remote_code': True, 'device_map': 'auto'}
    if torch.cuda.is_available():
        kwargs.update(
            {
                'torch_dtype': torch.float16,
                'quantization_config': BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type='nf4',
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                ),
            }
        )
    model = AutoModelForCausalLM.from_pretrained(model_path, **kwargs)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


def model_device(model) -> torch.device:
    return next(model.parameters()).device

In [ ]:
def labels_block() -> str:
    lines = []
    for label in LABELS:
        lines.append(f'- {label}: {LABEL_DESCRIPTIONS_RU[label]}')
    return '\n'.join(lines)


def truncate_text(text: str, max_chars: int = 900) -> str:
    text = ' '.join(str(text).split())
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(' ', 1)[0] + ' ...'


def build_few_shot_examples(train_df: pd.DataFrame, per_class: int = 1) -> str:
    examples = []
    for label in LABELS:
        class_rows = train_df[train_df['label_str'] == label].sort_values('sample_id').head(per_class)
        for _, row in class_rows.iterrows():
            examples.append(
                'Текст: ' + truncate_text(row[TEXT_COLUMN], 650) + '\n'
                'Ответ: {"label": "' + label + '"}'
            )
    return '\n\n'.join(examples)


def build_user_prompt(text: str, mode: str, few_shot_examples: str = '') -> str:
    base = f'''Задача: классифицировать русский аргумент по типу логической ошибки.

Допустимые метки:
{labels_block()}

Правила:
1. Выбери ровно одну метку из списка.
2. Если явной ошибки из списка нет, выбери "none".
3. Не добавляй новых классов.
4. Отвечай валидным JSON.

Текст:
{text}
'''

    if mode == 'zero_shot':
        return base + '\nФормат ответа: {"label": "<одна метка>"}'

    if mode == 'few_shot':
        return f'''Задача: классифицировать русский аргумент по типу логической ошибки.

Допустимые метки:
{labels_block()}

Примеры:
{few_shot_examples}

Теперь классифицируй новый текст.

Правила:
1. Выбери ровно одну метку из списка.
2. Если явной ошибки из списка нет, выбери "none".
3. Не добавляй новых классов.
4. Отвечай валидным JSON.

Текст:
{text}

Формат ответа: {{"label": "<одна метка>"}}'''

    if mode == 'reasoning':
        return base + '''\nПеред выбором проверь: есть ли вывод, какая опора используется для вывода, достаточно ли этой опоры.
Не расписывай длинную цепочку рассуждений. Дай краткое обоснование в одном предложении и итоговую метку.

Формат ответа: {"label": "<одна метка>", "rationale": "<одно короткое предложение>"}'''

    raise ValueError(f'Unknown prompt mode: {mode}')


def build_chat_prompt(tokenizer, text: str, mode: str, few_shot_examples: str = '') -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_prompt(text, mode, few_shot_examples=few_shot_examples)},
    ]
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return SYSTEM_PROMPT + '\n\n' + messages[-1]['content'] + '\n'

In [ ]:
def extract_json_object(text: str) -> dict[str, Any] | None:
    match = re.search(r'\{.*?\}', text, flags=re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def parse_generated_label(text: str) -> tuple[str, bool]:
    payload = extract_json_object(text)
    if isinstance(payload, dict) and 'label' in payload:
        candidate = normalize_label(payload.get('label', ''))
        if candidate in LABEL_TO_ID:
            return candidate, True

    normalized = normalize_label(text)
    normalized = normalized.replace('label:', '').replace('метка:', '').strip()
    for label in sorted(LABELS, key=len, reverse=True):
        if normalized == label or normalized.startswith(label):
            return label, True
    for label in sorted(LABELS, key=len, reverse=True):
        if label in normalized:
            return label, True
    return 'none', False


def compute_metrics(labels: list[int], preds: list[int], parse_ok: list[bool]) -> dict[str, Any]:
    label_order = list(range(N_CLASSES))
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=label_order,
        average=None,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'macro_precision': float(macro_precision),
        'macro_recall': float(macro_recall),
        'macro_f1': float(macro_f1),
        'parse_success_rate': float(np.mean(parse_ok)) if parse_ok else 0.0,
        'per_class_f1': {ID_TO_LABEL[idx]: float(value) for idx, value in zip(label_order, f1)},
        'classification_report': classification_report(
            labels,
            preds,
            labels=label_order,
            target_names=[ID_TO_LABEL[idx] for idx in label_order],
            output_dict=True,
            zero_division=0,
        ),
        'confusion_matrix': {
            'labels': [ID_TO_LABEL[idx] for idx in label_order],
            'matrix': confusion_matrix(labels, preds, labels=label_order).tolist(),
        },
    }

In [ ]:
def generate_for_mode(model, tokenizer, eval_df: pd.DataFrame, mode: str, few_shot_examples: str):
    prompts = [
        build_chat_prompt(tokenizer, row[TEXT_COLUMN], mode, few_shot_examples=few_shot_examples)
        for row in eval_df.to_dict('records')
    ]
    preds, parse_ok, raw_outputs = [], [], []
    device = model_device(model)

    for start in tqdm(range(0, len(prompts), EVAL_BATCH_SIZE), desc=f'Generate {mode}', leave=False):
        batch_prompts = prompts[start:start + EVAL_BATCH_SIZE]
        encoded = tokenizer(
            batch_prompts,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
            padding=True,
            return_tensors='pt',
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        new_tokens = generated[:, encoded['input_ids'].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        for output_text in decoded:
            label, ok = parse_generated_label(output_text)
            preds.append(LABEL_TO_ID[label])
            parse_ok.append(ok)
            raw_outputs.append(output_text.strip())

    return preds, parse_ok, raw_outputs


def evaluate_mode(model, tokenizer, eval_df: pd.DataFrame, mode: str, few_shot_examples: str, output_dir: Path):
    preds, parse_ok, raw_outputs = generate_for_mode(model, tokenizer, eval_df, mode, few_shot_examples)
    gold = eval_df['label_id'].astype(int).tolist()
    metrics = compute_metrics(gold, preds, parse_ok)
    rows = []
    for row, pred, raw, ok in zip(eval_df.to_dict('records'), preds, raw_outputs, parse_ok):
        rows.append(
            {
                'split': EVAL_SPLIT,
                'prompt_mode': mode,
                'sample_id': int(row['sample_id']),
                'gold_label_id': int(row['label_id']),
                'pred_label_id': int(pred),
                'gold_label': ID_TO_LABEL[int(row['label_id'])],
                'pred_label': ID_TO_LABEL[int(pred)],
                'raw_generation': raw,
                'parse_ok': bool(ok),
                'text_column': TEXT_COLUMN,
                'text': str(row[TEXT_COLUMN]),
                'source_dataset_path': str(DATASET_PATH),
            }
        )

    mode_dir = output_dir / mode
    mode_dir.mkdir(parents=True, exist_ok=True)
    write_json(mode_dir / 'metrics.json', {k: v for k, v in metrics.items() if k not in {'classification_report', 'confusion_matrix'}})
    write_json(mode_dir / 'classification_report.json', metrics['classification_report'])
    write_json(mode_dir / 'confusion_matrix.json', metrics['confusion_matrix'])
    write_jsonl(mode_dir / 'predictions.jsonl', rows)
    return metrics, rows

In [ ]:
set_global_seed(SEED)
df = load_cocolofa_jsonl(DATASET_PATH)
train_df = df[df['split'] == 'train'].reset_index(drop=True)
eval_df = df[df['split'] == EVAL_SPLIT].reset_index(drop=True)
if MAX_EVAL_ROWS is not None and len(eval_df) > MAX_EVAL_ROWS:
    eval_df = eval_df.sample(n=MAX_EVAL_ROWS, random_state=SEED).reset_index(drop=True)

print('dataset rows:', len(df))
print('eval rows:', len(eval_df))
print('eval split:', EVAL_SPLIT)
print('label distribution:', dict(eval_df['label_str'].value_counts().sort_index()))
print('text column:', TEXT_COLUMN)
display(eval_df[['sample_id', 'label_str', TEXT_COLUMN]].head(3))

In [ ]:
model_path = resolve_model_path()
print('resolved model path:', model_path)
model, tokenizer = load_model_and_tokenizer(model_path)
few_shot_examples = build_few_shot_examples(train_df, per_class=FEW_SHOT_PER_CLASS)
print('few-shot examples chars:', len(few_shot_examples))

In [ ]:
run_slug = re.sub(r'[^a-zA-Z0-9._-]+', '_', str(model_path).strip('/').split('/')[-1] or 'decoder_model')
output_dir = OUTPUT_ROOT / f'{run_slug}_text_ru_prompting_seed{SEED}'
output_dir.mkdir(parents=True, exist_ok=True)

write_json(output_dir / 'config.json', {
    'dataset_path': str(DATASET_PATH),
    'model_path': str(model_path),
    'kaggle_model_handle': KAGGLE_MODEL_HANDLE,
    'text_column': TEXT_COLUMN,
    'seed': SEED,
    'prompt_modes': PROMPT_MODES,
    'eval_split': EVAL_SPLIT,
    'max_eval_rows': MAX_EVAL_ROWS,
    'max_input_length': MAX_INPUT_LENGTH,
    'max_new_tokens': MAX_NEW_TOKENS,
    'few_shot_per_class': FEW_SHOT_PER_CLASS,
})

all_metrics = {}
all_rows = []
for mode in PROMPT_MODES:
    assert mode in {'zero_shot', 'few_shot', 'reasoning'}, mode
    metrics, rows = evaluate_mode(model, tokenizer, eval_df, mode, few_shot_examples, output_dir)
    all_metrics[mode] = {k: v for k, v in metrics.items() if k not in {'classification_report', 'confusion_matrix'}}
    all_rows.extend(rows)
    print(mode, all_metrics[mode])

write_json(output_dir / 'summary_metrics.json', all_metrics)
write_jsonl(output_dir / 'all_predictions.jsonl', all_rows)
all_metrics

In [ ]:
summary_df = pd.DataFrame([
    {'prompt_mode': mode, **metrics}
    for mode, metrics in all_metrics.items()
]).sort_values('macro_f1', ascending=False)
display(summary_df)

best_mode = summary_df.iloc[0]['prompt_mode']
cm_payload = read_json(output_dir / best_mode / 'confusion_matrix.json')
cm_df = pd.DataFrame(cm_payload['matrix'], index=cm_payload['labels'], columns=cm_payload['labels'])

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title(f'Decoder-only Prompting | {best_mode} | Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()

## Как запускать следующие режимы

Первый запуск оставьте `PROMPT_MODES = ['zero_shot']`.
После проверки `parse_success_rate` можно поставить:

```python
PROMPT_MODES = ['zero_shot', 'few_shot', 'reasoning']
```

`reasoning` здесь означает краткое структурное обоснование, а не длинный chain-of-thought.
Для метрик используется только поле `label` из JSON.